[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sterolose/sirius-statistics-2026/blob/main/notebooks/5_nonparametric.ipynb)

## Семинар 5. Непараметрические методы, мощность теста, минимально обнаруживаемый эффект

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

В этот раз используем датасет [Exercise](https://github.com/mwaskom/seaborn-data/blob/master/exercise.csv). Датасет содержит 90 измерений пульса 30 участников. Каждый участник выполнял только один вид активности, во время которой его пульс измеряли трижды: на 1-й, 15-й и 30-й минутах от начала. Участники также были разделены на две группы по диете.

Столбцы содержат следующие переменные:
- `id` — идентификатор участника;
- `diet` — диета: `low fat` и `no fat`;
- `pulse` — пульс, ударов в минуту;
- `time` — время измерения: `1 min`, `15 min`, `30 min`;
- `kind` — вид активности: `rest`, `walking`, `running`.

In [ ]:
df = sns.load_dataset("exercise")
df.head(2)

Удалим лишнюю колонку.

In [ ]:
df = df.drop(columns=df.columns[0])
df.head(2)

Посчитаем число наблюдений в каждой подгруппе

In [ ]:
df.value_counts(["kind", "diet", "time"]).to_frame(name="n")

Как обычно, во всех заданиях используем уровень значимости $\alpha = 0.05$ и двусторонние альтернативы.

### Задание 1

Постройте гистограмму распределения пульса для всей выборки.

In [ ]:
# Your code here

Постройте QQ-plot для оценки нормальности также по всему датасету.

In [ ]:
# Your code here

Выполните тест Шапиро — Уилка для подвыборок, образованных категориями переменной `kind`.

In [ ]:
# Your code here

### Задание 2

Построим диаграммы размаха пульса для разных видов активности и диеты.

In [ ]:
sns.boxplot(df, x="kind", y="pulse", hue="diet")
plt.show()

С помощью теста Манна — Уитни проверим, различается ли статистически пульс в покое и при ходьбе.

In [ ]:
rest = df.loc[df["kind"] == "rest", "pulse"]
walking = df.loc[df["kind"] == "walking", "pulse"]

result = stats.mannwhitneyu(rest, walking)
f"U = {result.statistic}, p-value = {result.pvalue:.3g}"

С помощью теста Манна — Уитни оцените, различается ли статистически пульс у людей с разной диетой. 

In [ ]:
# Your code here

Сравните с параметрическим аналогом.

In [ ]:
# Your code here

При каких условиях результат теста Манна — Уитни можно интерпретировать как различие медиан?

### Задание 3

Теперь построим диаграммы размаха пульса для разных видов активности и времени измерения.

In [ ]:
sns.boxplot(df, x="kind", y="pulse", hue="time")
plt.show()

Оценим, различается ли пульс через 15 и 30 минут у одного и того же человека, исключив группу покоя.  
Сперва отфильтруем датафрейм, а затем переведем его в "wide"-формат.

In [ ]:
group = df.loc[df["kind"] != "rest"]
group = df.pivot(
    index="id", columns="time", values="pulse",
)
group.head(2)

Проверьте, изменился ли систематически пульс с помощью парного теста Вилкоксона (`stats.wilcoxon`). Сравните результат с  параметрическим аналогом.

In [ ]:
# Your code here

### Задание 4

Предприятие внедряет новый производственный процесс, который должен снизить загрязнение близлежащего водоема. Концентрация загрязнителя в пробах воды считается нормально распределенной со стандартным отклонением **12 мг/л**. Средняя текущая концентрация составляет **80 мг/л**. Требуется спланировать исследование так, чтобы с вероятностью **80%** обнаружить изменение средней концентрации не менее чем на **20%** при уровне значимости **0.05**. Сколько проб нужно взять в контрольном эксперименте и при тестировании нового процесса, если сравнение проводится с помощью двухвыборочного t-теста? Ответ округлите вверх до целого числа.

Для t-теста в `statsmodels` существует специальный класс, который значительно упрощает решение.

In [ ]:
from statsmodels.stats.power import TTestIndPower

diff = 80 * 0.2
effect = diff / 12

size = TTestIndPower().solve_power(
    effect_size=effect,
    power=0.8,
    alpha=0.05,
)
int(np.ceil(size))

Какое минимальное количество проб необходимо взять для решения предыдущей задачи, если предположить, что концентрация распределена логнормально? Исходные параметры распределения (среднее и стандартное отклонение логарифма величины) принять равными **2.5** и **0.75**, а желаемое изменение первого — **0.5**.

Для такой постановки нет аналитического решения. Проведите симуляцию **1000** виртуальных экспериментов для каждого объема выборки, используя тест Манна — Уитни. Изобразите изменение частоты обнаружений значимого эффекта на графике.

In [ ]:
# Your code here